In [27]:
import akida
import pickle, os

import pandas as pd
import numpy as np

from dataset_NewEEG import filter_rawEEG

used_channels = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']

t_start = 0
t_end = 250

df = pd.read_csv("sample_eeg_data.csv", delimiter=',')
df = df.loc[:, used_channels]
X = np.array(df)[t_start:t_end].transpose(1,0)
                    
X = filter_rawEEG(X, 0.5, 35)


print("================================= \n\n Using Akida Neuromorphic Mode ...  \n")

chip = akida.devices()[0]



print(f"=> Akida chip detected = {chip} \n")


akida_model = akida.Model(os.path.join('saved_models', 'LR.fbz'))
with open(os.path.join('saved_models', 'cspLR.pkl'), 'rb') as f:
    csp = pickle.load(f)

X = np.expand_dims(X, 0)
X = csp.transform(X)
X = np.expand_dims(X, 3)


X = ((X - X.min()) / (X.max() - X.min()) * 255).astype(np.uint8)
X = np.pad(X, ((0, 0), (2, 2), (0, 0), (0, 0)), mode='constant', constant_values=0)

l = []
for i in range(1000):
    l.append(X.copy())
    
X = np.concatenate(l)
print(X.shape)

akida_model.map(chip, hw_only=True)
chip.soc.power_measurement_enabled = True
akida_model.predict(X)
print(akida_model.statistics)





 Using Akida Neuromorphic Mode ...  

=> Akida chip detected = <akida.core.HardwareDevice object at 0xffff4e601930> 

(1000, 7, 250, 1)
Average framerate = 944.29 fps
Last inference power range (mW):  Avg 913.71 / Min 907.00 / Max 936.00 / Std 9.55 
Last inference energy consumed (mJ/frame): 0.97


In [28]:
for pe in chip.soc.power_meter.events():
    print("power event time = ", pe.ts)
    print(f"current = {pe.current} mA")
    print(f"power =  {pe.power} mW")
    print(f"voltage =  {pe.voltage} microV")
    if pe.power != 900:
        print(pe.power)

power event time =  23737851
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23737922
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23737992
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738063
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738133
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738204
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738275
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738345
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738415
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738487
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738557
current = 1005 mA
power =  900 mW
voltage =  896250 microV
power event time =  23738627
cur